# 🐍 Olist ETL Pipeline
**Project:** Marketplace Analytics Pipeline — E-Commerce Sales & Operations  
**Input:** 9 raw CSVs
**Output:** 7 clean CSVs ready for SQL Server warehouse

In [1]:
import pandas as pd

folder = "D:/Olist/"

files = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv",
}

## Step 1 — Profile all 9 tables
Check row counts and null values before touching anything.

In [2]:
for name, f in files.items():
    df = pd.read_csv(folder + f)
    print(f"\n=== {name} ({len(df):,} rows) ===")
    print(df.isnull().sum()[df.isnull().sum() > 0])


=== orders (99,441 rows) ===
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

=== items (112,650 rows) ===
Series([], dtype: int64)

=== customers (99,441 rows) ===
Series([], dtype: int64)

=== sellers (3,095 rows) ===
Series([], dtype: int64)

=== products (32,951 rows) ===
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

=== payments (103,886 rows) ===
Series([], dtype: int64)

=== reviews (99,224 rows) ===
review_comment_title      87656
review_comment_message    58247
dtype: int64

=== geolocation (1,000,163 rows) ===
Series([], dtype: int64)

=== translation (71 rows) ===
Series([], dtype: int64)


**Expected findings:**
- `orders`: ~160 null delivered dates (cancelled orders) — normal
- `reviews`: ~145K null comments (star-only reviews) — normal
- `products`: ~600 missing dimensions, 2 missing category — needs fixing

## Step 2 — Clean Products
- Merge English category names (Portuguese → English)
- Label 2 uncategorized products as "unknown"
- Fill missing dimensions with median

In [3]:
# Load translation table
translation = pd.read_csv(folder + "product_category_name_translation.csv")
products = pd.read_csv(folder + "olist_products_dataset.csv")

# Join English names
products = products.merge(translation, on="product_category_name", how="left")

# 2 products have no category at all → label them
products["product_category_name_english"] = products["product_category_name_english"].fillna("unknown")

# Missing dimensions → fill with median (real products have typical sizes)
for col in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    products[col] = products[col].fillna(products[col].median())

products.to_csv(folder + "clean_products.csv", index=False)
print("products done:", len(products))

products done: 32951


## Step 3 — Clean Reviews
Null comments are normal (most reviewers just click a star).
Replace nulls with empty string, keep all rows — the score is the valuable part.

In [4]:
reviews = pd.read_csv(folder + "olist_order_reviews_dataset.csv")

reviews["review_comment_title"] = reviews["review_comment_title"].fillna("")
reviews["review_comment_message"] = reviews["review_comment_message"].fillna("")

reviews.to_csv(folder + "clean_reviews.csv", index=False)
print("reviews done:", len(reviews))

reviews done: 99224


## Step 4 — Clean Orders
- Convert 5 timestamp columns to datetime
- **Create 2 new KPI columns:**
  - `delivery_days` = purchase → delivered
  - `delivery_delay_days` = actual − estimated (negative = early, positive = late)

In [5]:
orders = pd.read_csv(folder + "olist_orders_dataset.csv")

date_cols = ["order_purchase_timestamp", "order_approved_at",
             "order_delivered_carrier_date", "order_delivered_customer_date",
             "order_estimated_delivery_date"]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# Delivery performance columns (future KPIs)
orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days
orders["delivery_delay_days"] = (orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]).dt.days

orders.to_csv(folder + "clean_orders.csv", index=False)
print("orders done:", len(orders))

orders done: 99441


## Step 5 — Aggregate Geolocation (1M rows → ~19K unique zips)
Raw table has many lat/lng points per zip code. Collapse to one row per zip.

In [6]:
geo = pd.read_csv(folder + "olist_geolocation_dataset.csv")

geo_agg = geo.groupby("geolocation_zip_code_prefix").agg(
    lat=("geolocation_lat", "mean"),
    lng=("geolocation_lng", "mean"),
    city=("geolocation_city", "first"),
    state=("geolocation_state", "first")
).reset_index()

geo_agg.to_csv(folder + "clean_geolocation.csv", index=False)
print("geolocation done:", len(geo), "→", len(geo_agg))

geolocation done: 1000163 → 19015


## Step 6 — Enrich Customers & Sellers with coordinates
Join zip → lat/lng so Power BI can map them later.

In [7]:
customers = pd.read_csv(folder + "olist_customers_dataset.csv")
sellers = pd.read_csv(folder + "olist_sellers_dataset.csv")

customers = customers.merge(geo_agg, left_on="customer_zip_code_prefix",
                            right_on="geolocation_zip_code_prefix", how="left")
customers = customers.drop(columns=["geolocation_zip_code_prefix"])

sellers = sellers.merge(geo_agg, left_on="seller_zip_code_prefix",
                        right_on="geolocation_zip_code_prefix", how="left")
sellers = sellers.drop(columns=["geolocation_zip_code_prefix"])

customers.to_csv(folder + "clean_customers.csv", index=False)
sellers.to_csv(folder + "clean_sellers.csv", index=False)
print("customers done:", len(customers), "| sellers done:", len(sellers))

customers done: 99441 | sellers done: 3095


## Step 7 — Items & Payments
Already clean — just copy to clean files (keeps naming consistent for SQL import).

In [8]:
items = pd.read_csv(folder + "olist_order_items_dataset.csv")
payments = pd.read_csv(folder + "olist_order_payments_dataset.csv")

items.to_csv(folder + "clean_items.csv", index=False)
payments.to_csv(folder + "clean_payments.csv", index=False)
print("items done:", len(items), "| payments done:", len(payments))

items done: 112650 | payments done: 103886


## Step 8 — Final Validation
Report card for every clean file.

In [9]:
import os

print("FINAL FILES CHECK")
for f in sorted(os.listdir(folder)):
    if f.startswith("clean_"):
        df = pd.read_csv(folder + f)
        nulls = df.isnull().sum().sum()
        print(f"{f}: {len(df):,} rows | total nulls: {nulls:,}")

FINAL FILES CHECK
clean_customers.csv: 99,441 rows | total nulls: 1,112
clean_geolocation.csv: 19,015 rows | total nulls: 0
clean_items.csv: 112,650 rows | total nulls: 0
clean_orders.csv: 99,441 rows | total nulls: 10,838
clean_payments.csv: 103,886 rows | total nulls: 0
clean_products.csv: 32,951 rows | total nulls: 2,440
clean_reviews.csv: 99,224 rows | total nulls: 145,903
clean_sellers.csv: 3,095 rows | total nulls: 28


## Step 9 — Insight Preview 📝
**These numbers go directly into resume bullets and the insights memo.**

In [10]:
total_rev = items["price"].sum() + items["freight_value"].sum()

delivered = orders.dropna(subset=["order_delivered_customer_date"])
on_time = (delivered["delivery_delay_days"] <= 0).mean() * 100

avg_score = reviews["review_score"].mean()
freight_pct = items["freight_value"].sum() / total_rev * 100
top_state = customers.groupby("customer_state").size().sort_values(ascending=False).index[0]

print(f"Total revenue: R$ {total_rev:,.0f}")
print(f"On-time delivery: {on_time:.1f}%")
print(f"Avg review score: {avg_score:.2f}")
print(f"Freight % of revenue: {freight_pct:.1f}%")
print(f"Top state by customers: {top_state}")

Total revenue: R$ 15,843,553
On-time delivery: 93.2%
Avg review score: 4.09
Freight % of revenue: 14.2%
Top state by customers: SP
